# Domain-adaptive pretraining — MLM trên Kanglish

**Settings:** Accelerator = **GPU T4 ×2** · Internet = **On** · ~25 phút

Từ vựng của MuRIL được xây cho tiếng Ấn viết bằng **chữ bản địa**, nên Kanglish chữ Latin bị xé
vụn: **2,05 mảnh/từ** so với **1,04** của tiếng Anh, và chỉ **36,7%** từ còn nguyên một mảnh.

```
government    ->  ['government']                        (tiếng Anh, 1 mảnh)
Bajetigu      ->  ['Ba', '##jet', '##ig', '##u']        (Kanglish, 4 mảnh)
```

Embedding của `##jet`, `##ikk` được học trong ngữ cảnh **chẳng liên quan gì** tới tiếng Kannada.
MLM kéo chúng về đúng chỗ — và vì nó **không cần nhãn**, nó tránh được đúng vấn đề đã giết chết
phương án nối dữ liệu ngoài vào train (đo được +0,0012 ± 0,0087, tức bằng không): *"offensive"*
khác *"hate"*, nhưng văn bản thì vẫn cùng một ngôn ngữ.

**Nói thẳng về quy mô:** kho của bạn ~**0,31 triệu token**, trong khi MuRIL pretrain trên ~16 **tỷ**
và các bài DAPT thường dùng 1–100 triệu. Thứ cứu vớt là chỉ **8.405 mục từ vựng (4,3%)** thực sự
xuất hiện, nên toàn bộ ngân sách dồn đúng vào những embedding đang sai. Kỳ vọng **+0,01 đến
+0,03**, không phải bước nhảy.

In [ ]:
import os, subprocess, sys

REPO, BRANCH, WORK = "trong5nhan6/Text", "main", "/kaggle/working"
TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
except Exception:
    pass
url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
hide = (lambda s: s.replace(TOKEN, "***")) if TOKEN else (lambda s: s)

os.chdir(WORK)
cmd = (["git", "-C", "repo", "pull", "--ff-only"] if os.path.isdir("repo/.git")
       else ["git", "clone", "--depth", "1", "-b", BRANCH, url, "repo"])
r = subprocess.run(cmd, capture_output=True, text=True)
print(hide((r.stdout + r.stderr).strip()))
if r.returncode:
    raise SystemExit("git that bai -- kiem tra Internet = On, repo Public")

os.chdir(f"{WORK}/repo"); sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip -q install ftfy sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1) Kho văn bản

Gom **mọi** file có Kanglish, khử trùng lặp. Không cần nhãn nào.

> **Lát held-out bị loại khỏi kho, mặc định.** Văn bản đó không mang nhãn nên giữ lại cũng không
> rò rỉ nhãn — nhưng model sẽ đã đọc đúng những câu ấy, và mọi macro-F1 đo trên lát đó sau này
> sẽ **lạc quan một cách âm thầm**. Giữ trung thực chỉ tốn 919 dòng trên 13.995.
>
> Ngược lại, `*_validation_inputs.csv` và file test **được** đưa vào: đó là transductive learning
> thông thường, và đúng là tình huống bản nộp sẽ chạy. Nhớ khai báo điều này trong bài báo.

In [ ]:
from pretrain_mlm import build_corpus
from src.utils.config import load_config
from src.data.preprocessing import ensure_processed

cfg = load_config("configs/base.yaml", task="a")
ensure_processed(cfg)
corpus = build_corpus(cfg, include_eval=False)
print(f"-> {len(corpus):,} dong")
for t in corpus[:5]:
    print("   ", t[:80])

## 2) Chạy MLM

`--epochs 15` trên ~13k dòng là khoảng 20–25 phút trên T4. Theo dõi **perplexity**: nó bắt đầu
cao vì các mảnh từ vựng đang sai với văn bản này, và **giảm xuống chính là sự thích nghi** mà
script này tồn tại để làm. Nếu nó không giảm, có gì đó sai.

In [ ]:
MODEL      = 'google/muril-base-cased'
MLM_EPOCHS = 15     # so epoch cua MLM. Fine-tune co so epoch RIENG, o muc 3.
BATCH      = 32     # TONG, chia deu cho cac GPU (2 x T4 -> 16/GPU). Giam neu OOM.
GRAD_ACCUM = 1      # BATCH x GRAD_ACCUM = batch hieu dung (32 x 1 = 32)
MAX_LEN    = 128    # do dai theo TOKEN; giam con 96 cung tiet kiem nhieu VRAM

!python pretrain_mlm.py --model {MODEL} --epochs {MLM_EPOCHS} --batch_size {BATCH} --grad_accum {GRAD_ACCUM} --max_len {MAX_LEN}

# Bien the:
# !python pretrain_mlm.py --model Hate-speech-CNERG/kannada-codemixed-abusive-MuRIL --epochs 15
# !python pretrain_mlm.py --model xlm-roberta-base --epochs 15
# !python pretrain_mlm.py --epochs 20 --lr 1e-4        # kho nho -> co the can lr cao hon
# !python pretrain_mlm.py --include_eval               # CHI cho ban nop cuoi, diem noi bo se lac quan
#
# NEU OOM: ha BATCH va tang GRAD_ACCUM de giu batch hieu dung (vd 16 x 2, hoac 8 x 4).
# CHI dung 1 GPU du co 2:  them --single_gpu
#
# THU NHANH truoc khi bo 25 phut GPU -- chay het ca phan luu trong ~1 phut:
# !python pretrain_mlm.py --max_rows 200 --epochs 1 --out /tmp/mlm_test

> **Về OOM.** Logits của MLM có shape `[batch, len, vocab]`, mà vocab của MuRIL là
> **197.285** — nên riêng một tensor đó ở `batch 32 × len 128` đã chiếm **3,2 GB**, và phải giữ
> hai bản (xuôi + ngược). Đó là thứ làm nổ T4, không phải model.
>
> | batch × len | logits (xuôi+ngược) | + model/AdamW | ~tổng |
> |---|---|---|---|
> | 32 × 128 | 6,46 GB | 3,81 GB | ~11,5 GB → **OOM** |
> | 16 × 128 | 3,23 GB | 3,81 GB | ~8,2 GB |
> | **8 × 128** | **1,62 GB** | 3,81 GB | **~6,6 GB** ✅ mặc định |
> | 4 × 128 | 0,81 GB | 3,81 GB | ~5,8 GB |
>
> **Trên 2×T4 script tự dùng cả hai GPU** (`DataParallel`), nên `BATCH` là **tổng** và mỗi GPU
> chỉ giữ một nửa:
>
> | BATCH tổng | /GPU | logits/GPU | **GPU 0** | GPU 1 |
> |---|---|---|---|---|
> | 16 | 8 | 1,62 GB | 6,4 GB | 3,6 GB |
> | **32** | **16** | **3,23 GB** | **8,0 GB** | 5,2 GB |
> | 48 | 24 | 4,85 GB | 9,7 GB | 6,8 GB |
> | 64 | 32 | 6,46 GB | 11,3 GB | 8,4 GB |
>
> GPU 0 luôn chật hơn vì chỉ nó giữ trọng số gốc, gradient và hai trạng thái AdamW (3,8 GB);
> GPU kia chỉ giữ một bản sao trọng số.
>
> **Chạy trên máy 1 GPU vẫn bình thường** — script tự dò `torch.cuda.device_count()` và chỉ bật
> `DataParallel` khi thấy nhiều hơn một. Checkpoint lưu ra giống hệt nhau trong cả hai trường hợp.
>
> Script in ước lượng VRAM **trước khi** train, nên bạn biết trước chứ không phải đợi nó crash.

## 3) Fine-tune từ checkpoint vừa thích nghi

Không cần code mới — chỉ trỏ `model.name` vào thư mục vừa lưu. Chạy **cả bản gốc lẫn bản MLM**
thì mới biết nó có giúp không.

In [ ]:
# (config, checkpoint MLM tuong ung). Them dong moi sau khi da chay pretrain_mlm.py cho model do.
PAIRS = [
    ('muril',   'checkpoints/mlm/muril-base-cased'),
  # ('roberta', 'checkpoints/mlm/xlm-roberta-base'),
  # ('cnerg_muril', 'checkpoints/mlm/kannada-codemixed-abusive-MuRIL'),
]
FT_EPOCHS = 6     # so epoch khi FINE-TUNE -- KHAC voi MLM_EPOCHS o tren (cai do la cua MLM).
                  # base.yaml dang de 20; doi gia tri thi BAT BUOC co suffix, nen no nam trong SUF.
SUF = f'_e{FT_EPOCHS}'
# patience = epochs tuc TAT early stopping: trainer van giu epoch tot nhat, con lich LR duoc
# anneal het. Voi epochs=20 thi model dat dinh o epoch ~5 luc LR con ~83% -- phi doan anneal.
FT = f'--set training.epochs={FT_EPOCHS} training.early_stopping_patience={FT_EPOCHS}'

# Ten thu muc TU PHAN BIET: run_name gan them '_mlm' khi model.name bi ghi de, nen ban goc va
# ban MLM khong bao gio dung chung mot thu muc, va khong can them --run_suffix bang tay.
for cfg, ckpt in PAIRS:
    for t in ('a', 'b'):
        print("=" * 70)
        !python train.py --config configs/{cfg}.yaml --task {t} {FT} --run_suffix {SUF}
        !python train.py --config configs/{cfg}.yaml --task {t} {FT} model.name={ckpt} --run_suffix {SUF}

In [ ]:
import pandas as pd
d = pd.read_csv('results/metrics.csv')
display(d[d.run.str.contains('muril')][['task', 'run', 'macro_f1', 'accuracy', 'best_epoch']]
        .sort_values(['task', 'macro_f1'], ascending=[True, False]))

## 4) Tải checkpoint về

**~0,9 GB** — chỉ tải nếu bạn muốn dùng lại ở máy hoặc ở session Kaggle khác. Nếu không, cứ chạy
lại notebook này (25 phút) thì nhanh hơn là tải lên tải xuống.

In [ ]:
!cd /kaggle/working/repo && zip -r -q /kaggle/working/mlm_muril.zip checkpoints/mlm
!ls -lh /kaggle/working/mlm_muril.zip